# digits/ -- dataset download (MNIST, USPS, SVHN)

Downloads the three raw datasets via `torchvision.datasets` and dumps each
train/test split to a plain numpy `.npz` cache under `digits/data/`, at each
dataset's native resolution and channel count:

- MNIST: 60,000 train + 10,000 test, 28x28 grayscale, 10 classes
- USPS: ~7,291 train + 2,007 test, 16x16 grayscale, 10 classes
- SVHN: ~73,257 train + ~26,032 test, 32x32 RGB, 10 classes

Uniform preprocessing across domains (grayscale conversion, resize to a
common resolution, source-only normalization) is deliberately NOT done
here -- it happens at load time in `digits/data_utils.py`, so these caches
always reflect the raw, unmodified data.

Idempotent: re-running this notebook will not re-download anything
`torchvision` already finds on disk with a matching checksum.

## Setup

In [1]:
import sys
from pathlib import Path

root_dir = Path().resolve().parent
sys.path.insert(0, str(root_dir))

import numpy as np
from torchvision import datasets

DATA_DIR = root_dir / "digits" / "data"
RAW_DIR = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)


def dump_split(dataset, out_path: Path, remap_svhn_ten_to_zero: bool = False) -> tuple:
    """dataset: any torchvision Dataset yielding (PIL Image, int label) pairs
    (transform=None). Stacks every image into one uint8 array and every
    label into one int64 array, saved together as a compressed .npz."""
    X = np.stack([np.array(img) for img, _ in dataset]).astype(np.uint8)
    y = np.array([label for _, label in dataset], dtype=np.int64)
    if remap_svhn_ten_to_zero:
        # Not actually needed: torchvision.datasets.SVHN.__init__ already
        # remaps label 10 -> 0 internally (verified directly in its source,
        # torchvision/datasets/svhn.py: `np.place(self.labels, self.labels
        # == 10, 0)`). This is therefore a no-op assertion, not a fix --
        # kept so the requirement "digit 0 must be labeled 0" is verified
        # rather than silently assumed.
        assert not (y == 10).any(), (
            "unexpected label 10 in SVHN labels -- torchvision's own remap "
            "(10 -> 0) should already have made this impossible"
        )
    np.savez_compressed(out_path, X=X, y=y)
    return X.shape, y.shape


print(f"Data directory: {DATA_DIR}")

Data directory: /Users/riccardo/Desktop/PML/BayesianExoAdaptation/digits/data


## MNIST (28x28 grayscale, 10 classes)

In [2]:
for split_name, train_flag in [("train", True), ("test", False)]:
    ds = datasets.MNIST(str(RAW_DIR), train=train_flag, download=True)
    shape = dump_split(ds, DATA_DIR / f"mnist_{split_name}.npz")
    print(f"mnist_{split_name}: X{shape[0]} y{shape[1]}")

mnist_train: X(60000, 28, 28) y(60000,)


mnist_test: X(10000, 28, 28) y(10000,)


## USPS (16x16 grayscale, 10 classes)

In [3]:
for split_name, train_flag in [("train", True), ("test", False)]:
    ds = datasets.USPS(str(RAW_DIR), train=train_flag, download=True)
    shape = dump_split(ds, DATA_DIR / f"usps_{split_name}.npz")
    print(f"usps_{split_name}: X{shape[0]} y{shape[1]}")

usps_train: X(7291, 16, 16) y(7291,)


usps_test: X(2007, 16, 16) y(2007,)


## SVHN (32x32 RGB, 10 classes)

Label "10" corresponds to digit "0" in the raw `.mat` files; verified
directly in `torchvision.datasets.svhn.SVHN.__init__` that it already
remaps this internally (`np.place(self.labels, self.labels == 10, 0)`) --
the assertion in `dump_split` above checks this rather than assuming it.

Note: `ufldl.stanford.edu` (SVHN's host) has been observed to stall
mid-download in this environment (plain HTTP, no timeout in
`torchvision`'s own downloader). If the cell below hangs with no progress
for more than ~1-2 minutes, interrupt it and download the stalled `.mat`
file directly with a resumable, stall-aware retry loop instead, e.g. for
`train_32x32.mat` (repeat for `test_32x32.mat`, expected sizes
182,040,794 / 64,275,384 bytes):

```bash
for i in $(seq 1 15); do
  curl -fSL -C - --connect-timeout 15 --max-time 200 --speed-time 20 --speed-limit 1000 \
    -o digits/data/raw/train_32x32.mat \
    "http://ufldl.stanford.edu/housenumbers/train_32x32.mat" && break
done
```

Then re-run the cell below -- `torchvision` will find the already-downloaded,
checksummed `.mat` file and skip straight to the `.npz` conversion.

In [4]:
for split_name in ["train", "test"]:
    ds = datasets.SVHN(str(RAW_DIR), split=split_name, download=True)
    shape = dump_split(ds, DATA_DIR / f"svhn_{split_name}.npz", remap_svhn_ten_to_zero=True)
    print(f"svhn_{split_name}: X{shape[0]} y{shape[1]}")

Using downloaded and verified file: /Users/riccardo/Desktop/PML/BayesianExoAdaptation/digits/data/raw/train_32x32.mat


svhn_train: X(73257, 32, 32, 3) y(73257,)
Using downloaded and verified file: /Users/riccardo/Desktop/PML/BayesianExoAdaptation/digits/data/raw/test_32x32.mat


svhn_test: X(26032, 32, 32, 3) y(26032,)


## Final image counts

In [5]:
for name in ["mnist_train", "mnist_test", "usps_train", "usps_test",
             "svhn_train", "svhn_test"]:
    d = np.load(DATA_DIR / f"{name}.npz")
    labels_present = sorted(set(d["y"].tolist()))
    print(f"  {name:12s}: {d['X'].shape[0]:6d} images, shape {d['X'].shape[1:]}, "
          f"labels {labels_present}")

  mnist_train :  60000 images, shape (28, 28), labels [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
  mnist_test  :  10000 images, shape (28, 28), labels [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
  usps_train  :   7291 images, shape (16, 16), labels [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
  usps_test   :   2007 images, shape (16, 16), labels [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


  svhn_train  :  73257 images, shape (32, 32, 3), labels [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


  svhn_test   :  26032 images, shape (32, 32, 3), labels [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
